<a href="https://colab.research.google.com/github/kessa8691-sudo/assignment-no-1-/blob/main/assignment%202%20task%202.0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import kagglehub
path = kagglehub.dataset_download("camnugent/california-housing-prices")

100%|██████████| 400k/400k [00:00<00:00, 78.0MB/s]

Extracting files...


In [4]:
import kagglehub
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import joblib

#STEP 1: DATA LOADING & OVERVIEW

In [5]:
print("--- Step 1: Loading Data ---")
path = kagglehub.dataset_download("camnugent/california-housing-prices")

--- Step 1: Loading Data ---
Using Colab cache for faster access to the 'california-housing-prices' dataset.


In [6]:
csv_file_path = os.path.join(path, "housing.csv")

In [7]:
csv_file_path = os.path.join(path, "housing.csv")

In [8]:
df = pd.read_csv(csv_file_path)

# STEP 2: DATA ANALYSIS / MINING

In [9]:
print("\n--- Step 2: Data Analysis ---")
print(f"Dataset Shape: {df.shape}")
print("\nChecking for missing values:")


--- Step 2: Data Analysis ---
Dataset Shape: (20640, 10)

Checking for missing values:


In [10]:
print(df.isnull().sum())

longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64


In [11]:
# Define features (X) and target (y)
X = df.drop('median_house_value', axis=1)
y = df['median_house_value']

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# STEP 3 & 4: PRE-PROCESSING, IMPUTATION & FEATURE SELECTION

In [13]:
print("\n--- Step 3 & 4: Setting up Pre-processing Flow ---")
# We must treat Numbers and Text (Categorical) differently.

# 3A. Identify numerical and categorical columns automatically
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X_train.select_dtypes(include=['object']).columns


--- Step 3 & 4: Setting up Pre-processing Flow ---


In [24]:
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [25]:
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')) # Converts text like "NEAR BAY" to 1s and 0s
])

In [26]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# STEP 5: MODEL SELECTION & TRAINING

In [28]:
print("\n--- Step 5: Model Training ---")
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])


--- Step 5: Model Training ---


In [29]:
model_pipeline.fit(X_train, y_train)
print("Model trained successfully.")

Model trained successfully.


In [30]:
print("\n--- Step 6: Evaluation & Bias-Variance Check ---")
train_predictions = model_pipeline.predict(X_train)
test_predictions = model_pipeline.predict(X_test)


--- Step 6: Evaluation & Bias-Variance Check ---


In [32]:
train_r2 = r2_score(y_train, train_predictions)
test_r2 = r2_score(y_test, test_predictions)

print(f"Training R2 Score: {train_r2:.4f}")
print(f"Testing R2 Score:  {test_r2:.4f}")

if train_r2 > test_r2 + 0.10:
    print("Insight: Model might be overfitting (High Variance).")
elif train_r2 < 0.60:
    print("Insight: Model might be underfitting (High Bias). Linear Regression may be too simple for this data.")
else:
    print("Insight: Model shows a good Bias-Variance balance.")

Training R2 Score: 0.6497
Testing R2 Score:  0.6254
Insight: Model shows a good Bias-Variance balance.


# STEP 7: DEPLOYMENT READINESS

In [33]:
print("\n--- Step 7: Deployment ---")
joblib.dump(model_pipeline, 'kaggle_california_pipeline.pkl')
print("Model pipeline saved as 'kaggle_california_pipeline.pkl' for deployment.")


--- Step 7: Deployment ---
Model pipeline saved as 'kaggle_california_pipeline.pkl' for deployment.


In [34]:
print("\n--- Step 8: Inference (Real-World Prediction) ---")
loaded_model = joblib.load('kaggle_california_pipeline.pkl')


--- Step 8: Inference (Real-World Prediction) ---


In [35]:
new_house = pd.DataFrame([{
    'longitude': -122.23,
    'latitude': 37.88,
    'housing_median_age': 41.0,
    'total_rooms': 880.0,
    'total_bedrooms': 129.0,
    'population': 322.0,
    'households': 126.0,
    'median_income': 8.3252,
    'ocean_proximity': 'NEAR BAY' # The pipeline will automatically One-Hot encode this!
}])

print("New House Data:")
print(new_house.to_string(index=False))

New House Data:
 longitude  latitude  housing_median_age  total_rooms  total_bedrooms  population  households  median_income ocean_proximity
   -122.23     37.88                41.0        880.0           129.0       322.0       126.0         8.3252        NEAR BAY


In [37]:
predicted_price = loaded_model.predict(new_house)[0]
print(f"\n Prediction Result: The model predicts this house is worth ${predicted_price:,.2f}")


 Prediction Result: The model predicts this house is worth $410,584.36


In [38]:
# Interactive Prediction
print("\n--- Interactive House Price Prediction ---")

# Collect user input for each feature
user_longitude = float(input("Enter longitude (e.g., -122.23): "))
user_latitude = float(input("Enter latitude (e.g., 37.88): "))
user_housing_median_age = float(input("Enter housing median age (e.g., 41.0): "))
user_total_rooms = float(input("Enter total rooms (e.g., 880.0): "))
user_total_bedrooms = float(input("Enter total bedrooms (e.g., 129.0): "))
user_population = float(input("Enter population (e.g., 322.0): "))
user_households = float(input("Enter households (e.g., 126.0): "))
user_median_income = float(input("Enter median income (e.g., 8.3252): "))
user_ocean_proximity = input("Enter ocean proximity (<1H OCEAN, INLAND, NEAR OCEAN, NEAR BAY, ISLAND): ")
user_house_data = pd.DataFrame([{
    'longitude': user_longitude,
    'latitude': user_latitude,
    'housing_median_age': user_housing_median_age,
    'total_rooms': user_total_rooms,
    'total_bedrooms': user_total_bedrooms,
    'population': user_population,
    'households': user_households,
    'median_income': user_median_income,
    'ocean_proximity': user_ocean_proximity
}])
print("\nYour entered data:")
print(user_house_data.to_string(index=False))
user_predicted_price = loaded_model.predict(user_house_data)[0]
print(f"\nPrediction Result: The model predicts this house is worth ${user_predicted_price:,.2f}")


--- Interactive House Price Prediction ---
Enter longitude (e.g., -122.23): 15246
Enter latitude (e.g., 37.88): 56.25
Enter housing median age (e.g., 41.0): 1.60
Enter total rooms (e.g., 880.0): 625
Enter total bedrooms (e.g., 129.0): 1
Enter population (e.g., 322.0): 36
Enter households (e.g., 126.0): 95
Enter median income (e.g., 8.3252): 36
Enter ocean proximity (<1H OCEAN, INLAND, NEAR OCEAN, NEAR BAY, ISLAND): sand

Your entered data:
 longitude  latitude  housing_median_age  total_rooms  total_bedrooms  population  households  median_income ocean_proximity
   15246.0     56.25                 1.6        625.0             1.0        36.0        95.0           36.0            sand

Prediction Result: The model predicts this house is worth $-411,443,155.70
